# conditional-hparam-branch — worked example 3: Register a LayerNorm submodule only when use_norm is set

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conditional-hparam-branch`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A block can optionally normalize its output. The clean pattern mirrors `nn.Linear`'s optional bias: set `self.norm = nn.LayerNorm(d)` when enabled and `self.norm = None` otherwise, then gate the forward call on `if self.norm is not None`. Setting the attribute to `None` (rather than skipping it) means `forward` can always reference `self.norm` without an `AttributeError`.

## Worked solution

1. **Always set the attribute.** In `__init__`, after `super().__init__()` and the `nn.Linear`, branch on the flag: `if use_norm: self.norm = nn.LayerNorm(d) else: self.norm = None`. Both branches assign `self.norm`, so it always exists.
2. **Why `None` and not skip?** If you skipped the assignment in the off case, `self.norm` would be undefined and `forward` would raise `AttributeError`. Assigning `None` lets the same `forward` code handle both configs.
3. **Gate the forward.** `if self.norm is not None: x = self.norm(x)`. When norm is off this is a clean pass-through of the linear output.
4. **Parameter accounting.** With norm on, `named_parameters()` contains `norm.weight` and `norm.bias`; with it off, those names are absent — `None` is not a Module, so PyTorch registers no parameters for it. This keeps the two configurations' parameter sets correctly different.

In [ ]:
class MLPBlock(t.nn.Module):
    def __init__(self, d, use_norm):
        super().__init__()
        self.fc = t.nn.Linear(d, d)
        if use_norm:
            self.norm = t.nn.LayerNorm(d)
        else:
            self.norm = None

    def forward(self, x):
        x = self.fc(x)
        if self.norm is not None:
            x = self.norm(x)
        return x

t.manual_seed(0)
block_on = MLPBlock(4, use_norm=True)
block_off = MLPBlock(4, use_norm=False)
print('use_norm=True params: ', sorted(dict(block_on.named_parameters()).keys()))
print('use_norm=False params:', sorted(dict(block_off.named_parameters()).keys()))
print('block_off.norm is None:', block_off.norm is None)